<a href="https://colab.research.google.com/github/noor486/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noor486/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/noor486/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

os.makedirs("work/outputs", exist_ok=True)
print("Working dir:", os.getcwd())

Working dir: /content/flyrank-ml-internship/flyrank-ml-internship


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 1. Method Choice and Why

**Method: Random Forest classifier**, predicting `is_declining_label`.

Why this over alternatives:
- Logistic Regression would be simpler and more explainable, but my Week 1
  discovery already showed several signals (search_volume vs. traffic) have
  weak or no linear relationship -- a linear model is likely to underfit
  real patterns here.
- A single Decision Tree (used in Notebook 2) is readable but overfits easily
  and is unstable to small data changes.
- Random Forest averages many trees, reducing overfitting versus a single
  tree, while still handling non-linear relationships Logistic Regression
  would miss. It's also what the reference pipeline already validated at
  Precision@50 = 0.740 -- using it here lets me reproduce and interpret
  that result myself, on the same features I built in Week 4, rather than
  trust it blind.
- I am NOT reaching for Gradient Boosting here -- added complexity isn't
  earning its keep yet given Random Forest already clears the baseline by
  a wide margin; adding a fancier model without first checking if the
  simpler one already works would be complexity for its own sake.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

features = ["search_volume", "competition", "cpc", "word_count",
            "impressions_90d", "sessions_90d", "content_age_days",
            "days_since_last_update", "ctr", "avg_position"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"]
groups = df["client_id"]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

train_clients = set(df.iloc[train_idx]["client_id"])
test_clients = set(df.iloc[test_idx]["client_id"])
print(f"Train rows: {len(X_train)}, Test rows: {len(X_test)}")
print(f"Client overlap between train/test: {len(train_clients & test_clients)} (should be 0)")

Train rows: 22885, Test rows: 7115
Client overlap between train/test: 0 (should be 0)


## 2. Split Design

**Client-holdout split** -- entire clients go to either train or test, never
both. This matters because pages from the same client share site structure
and baseline traffic patterns; a random row-level split would leak client
identity and inflate the score, exactly the leakage lesson from Notebook 2.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)
rf_proba = rf.predict_proba(X_test)[:, 1]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# Rebuild my Week 4 baseline rule score on the SAME test split
test_df = df.iloc[test_idx].copy()
test_df["is_stale"] = test_df["days_since_last_update"] >= 180
test_df["is_visible"] = test_df["impressions_90d"] >= 500
test_df["baseline_score"] = test_df["impressions_90d"].where(
    test_df["is_stale"] & test_df["is_visible"], 0
)

for k in (20, 50):
    base_p = precision_at_k(test_df["baseline_score"].values, y_test.values, k)
    rf_p = precision_at_k(rf_proba, y_test.values, k)
    print(f"Precision@{k}:  baseline = {base_p:.3f}   |   random forest = {rf_p:.3f}")

Precision@20:  baseline = 0.500   |   random forest = 0.600
Precision@50:  baseline = 0.620   |   random forest = 0.640


**Model-vs-baseline table:**

| Metric | Baseline (Week 4) | Random Forest |
|---|---|---|
| Precision@20 | 0.500 | 0.600 |
| Precision@50 | 0.620 | 0.640 |

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.inspection import permutation_importance

perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42, scoring="roc_auc")
importance_df = pd.DataFrame({
    "feature": features,
    "importance": perm.importances_mean
}).sort_values("importance", ascending=False)
print(importance_df)

                  feature  importance
4         impressions_90d    0.064625
6        content_age_days    0.022950
9            avg_position    0.011754
8                     ctr    0.011146
5            sessions_90d    0.005204
2                     cpc   -0.000115
1             competition   -0.001064
0           search_volume   -0.001095
3              word_count   -0.003948
7  days_since_last_update   -0.012491


In [ ]:
# Look at the model's biggest misses: high-confidence wrong predictions
test_df["rf_proba"] = rf_proba
test_df["actual"] = y_test.values
false_positives = test_df[(test_df["rf_proba"] > 0.7) & (test_df["actual"] == 0)]
false_negatives = test_df[(test_df["rf_proba"] < 0.3) & (test_df["actual"] == 1)]
print(f"High-confidence false positives: {len(false_positives)}")
print(f"High-confidence false negatives: {len(false_negatives)}")
false_positives[features + ["rf_proba"]].head(5)

High-confidence false positives: 412
High-confidence false negatives: 260


,search_volume,competition,cpc,word_count,impressions_90d,sessions_90d,content_age_days,days_since_last_update,ctr,avg_position,rf_proba
13,10.0,0.0,0.0,1342.0,307,4,238,103,0.00,39.8,0.753154
34,0.0,0.0,0.0,2542.0,3998,5,140,8,0.03,6.4,0.718899
78,0.0,0.0,0.0,1589.0,59,3,174,92,0.00,8.7,0.752377
126,0.0,0.0,0.0,2492.0,82,2,126,20,0.00,21.0,0.731238
134,0.0,0.0,0.0,2329.0,717,3,144,20,0.14,4.2,0.722387


**Errors and interpretation:** `impressions_90d` dominates permutation importance
by a wide margin (0.065, ~3x the next feature). The most notable finding:
`days_since_last_update` -- the entire basis of my Week 4 baseline rule --
has *negative* importance (-0.012) for the Random Forest, meaning it's not
just weak but actively unhelpful once other signals are available. The model
found a different path to the same prediction than my hand-written rule did.
`search_volume`, `competition`, and `word_count` are all near-zero, consistent
with my Week 1 discovery that these carry little real signal.

Errors: 412 high-confidence false positives vs. 260 high-confidence false
negatives -- the model is more prone to confidently-wrong "declining" calls
than to missing real decliners. A reviewer using this queue would waste more
time on false alarms than they'd miss real problems, which is worth
disclosing honestly rather than only reporting the aggregate precision number.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.